In [1]:
import numpy as np

X = np.load("../processed/X.npy")
y = np.load("../processed/y.npy")

print(X.shape)
print(y.shape)

(4635, 7, 297)
(4635,)


In [2]:
from sklearn.preprocessing import StandardScaler

X_flat = X.reshape(-1, 297)

scaler = StandardScaler()

X_flat = scaler.fit_transform(X_flat)

X = X_flat.reshape(
    X.shape[0],
    X.shape[1],
    X.shape[2]
)

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import Model

In [5]:
inputs = Input(shape=(7,297))

x = LSTM(
    256,
    return_sequences=True
)(inputs)

x = Dropout(0.2)(x)

x = LSTM(
    256,
    return_sequences=True
)(x)

x = Dropout(0.1)(x)

x = LSTM(
    256,
    return_sequences=False
)(x)

x = Dropout(0.2)(x)

outputs = Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(
    inputs,
    outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 7, 297)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 7, 256)         │       567,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,618,177 (6.17 MB)

 Trainable params: 1,618,177 (6.17 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [8]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 12s 73ms/step - accuracy: 0.5904 - loss: 0.6604 - val_accuracy: 0.6334 - val_loss: 0.6093
Epoch 2/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.7399 - loss: 0.5165 - val_accuracy: 0.6900 - val_loss: 0.5771
Epoch 3/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - accuracy: 0.8535 - loss: 0.3411 - val_accuracy: 0.6712 - val_loss: 0.6878
Epoch 4/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - accuracy: 0.9182 - loss: 0.1955 - val_accuracy: 0.6631 - val_loss: 0.8335
Epoch 5/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - accuracy: 0.9577 - loss: 0.1210 - val_accuracy: 0.6900 - val_loss: 1.0106
Epoch 6/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 0.9661 - loss: 0.0888 - val_accuracy: 0.7116 - val_loss: 1.0155
Epoch 7/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 0.9838 - loss: 0.0428 - val_accuracy: 0.6792 - val_loss: 1.1562


In [9]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

pred = (pred > 0.5).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
              precision    recall  f1-score   support

           0       0.73      0.66      0.69       467
           1       0.69      0.75      0.72       460

    accuracy                           0.71       927
   macro avg       0.71      0.71      0.71       927
weighted avg       0.71      0.71      0.70       927

